In [ ]:
#| default_exp azure

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from fastcore.all import *
import pulumi
import pulumi_azure_native as azure
from typing import Optional, List, Dict, Any
from pathlib import Path

## Storage

Azure Storage Account with security defaults

In [ ]:
#| export
@delegates(azure.storage.StorageAccount.__init__)
def storage(name:str=None, # Storage account name
            resource_group:str=None, # Resource group name
            location:str="eastus", # Azure region
            https_only:bool=True, # Require HTTPS
            min_tls:str="TLS1_2", # Minimum TLS version
            **kwargs):
    "Create Azure Storage Account with security defaults"
    
    # Create resource group if not provided
    if not resource_group:
        rg_name = f"pullup-{pulumi.get_project()}-{pulumi.get_stack()}"
        rg = azure.resources.ResourceGroup(rg_name, location=location)
        resource_group = rg.name
    
    account_name = name or f"pullup{pulumi.get_stack()}".replace("-", "")[:24]
    
    # Create storage account
    account = azure.storage.StorageAccount(account_name,
        resource_group_name=resource_group,
        location=location,
        sku=azure.storage.SkuArgs(
            name=azure.storage.SkuName.STANDARD_LRS),
        kind=azure.storage.Kind.STORAGE_V2,
        enable_https_traffic_only=https_only,
        minimum_tls_version=min_tls,
        allow_blob_public_access=False,
        encryption=azure.storage.EncryptionArgs(
            services=azure.storage.EncryptionServicesArgs(
                blob=azure.storage.EncryptionServiceArgs(enabled=True),
                file=azure.storage.EncryptionServiceArgs(enabled=True)),
            key_source=azure.storage.KeySource.MICROSOFT_STORAGE),
        **kwargs)
    
    return account

## Virtual Network

Azure VNet with subnets

In [ ]:
#| export
@delegates(azure.network.VirtualNetwork.__init__)
def vnet(name:str=None, # VNet name
         resource_group:str=None, # Resource group name
         location:str="eastus", # Azure region
         cidr:str="10.0.0.0/16", # Address space
         subnets:int=2, # Number of subnets
         **kwargs):
    "Create Azure Virtual Network with subnets"
    
    # Create resource group if not provided
    if not resource_group:
        rg_name = f"pullup-{pulumi.get_project()}-{pulumi.get_stack()}"
        rg = azure.resources.ResourceGroup(rg_name, location=location)
        resource_group = rg.name
    
    vnet_name = name or f"pullup-{pulumi.get_project()}"
    
    # Create subnet configurations
    subnet_configs = []
    for i in range(subnets):
        subnet_configs.append(azure.network.SubnetArgs(
            name=f"{vnet_name}-subnet-{i}",
            address_prefix=f"10.0.{i}.0/24"))
    
    # Create virtual network
    network = azure.network.VirtualNetwork(vnet_name,
        resource_group_name=resource_group,
        location=location,
        address_space=azure.network.AddressSpaceArgs(
            address_prefixes=[cidr]),
        subnets=subnet_configs,
        **kwargs)
    
    return network

## Container Instance

Azure Container Instance deployment

In [ ]:
#| export
@delegates(azure.containerinstance.ContainerGroup.__init__)
def container(name:str=None, # Container group name
              image:str="nginx:latest", # Docker image
              resource_group:str=None, # Resource group name
              location:str="eastus", # Azure region
              cpu:float=1.0, # CPU cores
              memory:float=1.5, # Memory in GB
              port:int=80, # Container port
              **kwargs):
    "Create Azure Container Instance"
    
    # Create resource group if not provided
    if not resource_group:
        rg_name = f"pullup-{pulumi.get_project()}-{pulumi.get_stack()}"
        rg = azure.resources.ResourceGroup(rg_name, location=location)
        resource_group = rg.name
    
    container_name = name or f"pullup-{pulumi.get_project()}"
    
    # Create container group
    container_group = azure.containerinstance.ContainerGroup(container_name,
        resource_group_name=resource_group,
        location=location,
        os_type=azure.containerinstance.OperatingSystemTypes.LINUX,
        containers=[azure.containerinstance.ContainerArgs(
            name=container_name,
            image=image,
            resources=azure.containerinstance.ResourceRequirementsArgs(
                requests=azure.containerinstance.ResourceRequestsArgs(
                    cpu=cpu,
                    memory_in_gb=memory)),
            ports=[azure.containerinstance.ContainerPortArgs(port=port)])],
        ip_address=azure.containerinstance.IpAddressArgs(
            type=azure.containerinstance.ContainerGroupIpAddressType.PUBLIC,
            ports=[azure.containerinstance.PortArgs(
                port=port,
                protocol=azure.containerinstance.ContainerGroupNetworkProtocol.TCP)]),
        **kwargs)
    
    return container_group

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()